# Laboratorul 5 — Exerciții
**Sisteme bazate pe cunoaștere · Extragerea și selecția trăsăturilor, primul model**

Aplicați instrumentele din `Lab5_Exemple_Rezolvate.ipynb` pe date **noi** și **justificați** fiecare alegere (o lungime de fereastră, o bandă, o adâncime, o penalizare) cu dovezi: un tabel, un grafic, un scor pe date nevăzute.

* **Partea 1 – De la semnal la un tabel de trăsături** (5 exerciții): ferestre, regula de etichetare, lungimea ferestrei, date reale, scurgere de informație prin suprapunere.
* **Partea 2 – Trei familii de trăsături** (9 exerciții): trăsături statistice, de frecvență și wavelet; aliere; ce bandă poartă defectul; ciclul zilnic; spectrul senzorului de lumină; tipuri și niveluri wavelet pe senzorul de sunet.
* **Partea 3 – Selecția trăsăturilor** (9 exerciții): filtre, redundanță, RFE, L1, scurgere prin selecție, stabilitate; analiza corelației, RFE și Lasso pe datele reale.
* **Partea 4 – Primul vostru model** (8 exerciții): linii de bază, diagnostic, adâncime, regularizare, matrice de confuzie, regresie, o linie de bază pentru prognoză, împărțire aleatorie vs pe grupuri.
* **Partea 5 – Mini-proiect** (1 temă deschisă): un tabel de rezultate cu linii de bază, modele și variație pe datele reale de ocupare.
* **Partea 6 – Un al doilea set de date: Smart Home Energy Usage** (2 exerciții): extragerea și selecția trăsăturilor pe un set în care răspunsul cinstit poate fi *„nu există nimic de găsit”*.

Fișiere necesare lângă caiet: `smart_office_dataset.csv`, `Occupancy_Estimation.csv`, `smart_home_energy_usage_dataset.csv` (Partea 6 citește doar primele 50 000 de rânduri). Biblioteci: `numpy pandas scipy scikit-learn matplotlib seaborn PyWavelets` (`pip install PyWavelets` dacă lipsește).

Dificultate: ★☆☆ încălzire · ★★☆ standard · ★★★ provocare. Autoverificările afișează ✅ sau ❌ + un indiciu. Numerele sunt comparate **după rotunjirea la 2 zecimale**, iar amprentele sunt aceleași pentru toți.

Fiecare exercițiu se încheie cu **Gândiți** sau **Scrieți**: răspundeți în celula markdown *„Interpretarea voastră”*; explicația **de ce** o valoare este bună valorează mai mult decât valoarea. **Nu modificați** celulele `Autoverificare` și nici celulele ajutătoare.

**Plan sugerat.** În cele 2 ore de laborator parcurgeți caietul cu exemple rezolvate și începeți exercițiile 1.1–1.3, 2.1–2.2, 4.1–4.2. Restul este tema (o săptămână): Partea 1–4, mini-proiectul 5.1 și, pentru mai multă practică, Partea 6.

**Predare:** *Kernel → Restart & Run All*, asigurați-vă că toate verificările sunt ✅ și că **răspunsurile scrise** (celulele markdown) și graficele sunt prezente. Predați fișierul `.ipynb` cu numele `Lab5_Prenume_Nume.ipynb`.

In [ ]:
import hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal, stats
import pywt

sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
PINK, BLUE, GREY, GREEN, BURG = "#A33573", "#2A7F9E", "#9AA0A6", "#1C7C54", "#4D1434"
plt.rcParams.update({"figure.dpi": 100, "font.size": 12, "axes.titlesize": 13, "axes.labelsize": 12,
                     "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 11,
                     "axes.titleweight": "bold", "legend.frameon": False})
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge, Lasso, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut, KFold, StratifiedKFold, cross_val_predict, cross_val_score, GridSearchCV
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif, RFE, VarianceThreshold, SequentialFeatureSelector
from sklearn.metrics import balanced_accuracy_score as bacc, accuracy_score, confusion_matrix, mean_absolute_error as mae

def _norm(v):
    if isinstance(v, (bool, np.bool_)): return str(bool(v))
    if isinstance(v, (int, float, np.integer, np.floating)): return f"{float(v):.2f}"
    return str(v)

def check(label, condition, hint=""):
    """Autoverificare: bifă verde dacă condiția este îndeplinită, altfel un indiciu. Nu modificați."""
    print(("✅ " if condition else "❌ ") + label + ("" if condition else f"   → indiciu: {hint}"))

def check_value(label, value, code, hint=""):
    """Compară valoarea voastră (numerele rotunjite la 2 zecimale) cu o amprentă a valorii așteptate."""
    ok = hashlib.sha256(_norm(value).encode()).hexdigest()[:8] == code
    print(("✅ " if ok else "❌ ") + label + ("" if ok else f"   → indiciu: {hint}"))

# ---------- funcții ajutătoare și generatoare de date folosite de exerciții (nu le modificați) ----------
FS_B = 2000   # frecvența de eșantionare a simulatorului de rulment (Hz)

SHORT = dict(temp_c="tin", humidity_pct="hum", co2_ppm="co2", temp_out_c="tout")

def load_office(path="smart_office_dataset.csv"):
    """Date smart office, curățate ca în Lab 4: codurile de eroare -> NaN, apoi golurile scurte se interpolează."""
    d = pd.read_csv(path, parse_dates=["timestamp"]).set_index("timestamp")
    d.loc[d.temp_c < 0, "temp_c"] = np.nan
    d.loc[(d.co2_ppm < 300) | (d.co2_ppm > 1500), "co2_ppm"] = np.nan
    cols = ["temp_c", "humidity_pct", "co2_ppm"]
    d[cols] = d[cols].interpolate(limit=60, method="time")
    return d

def stat_features(w):
    """Trăsăturile statistice ale unei ferestre (vector 1-D)."""
    w = np.asarray(w, float); n = len(w); t = np.arange(n); x = w - w.mean(); s = w.std(ddof=1)
    ac1 = np.corrcoef(w[:-1], w[1:])[0, 1] if s > 0 else 0.0
    return dict(mean=w.mean(), std=s, min=w.min(), max=w.max(), median=np.median(w), rms=np.sqrt(np.mean(w ** 2)),
                iqr=np.subtract(*np.percentile(w, [75, 25])), skew=stats.skew(w) if s > 0 else 0.0,
                kurt=stats.kurtosis(w) if s > 0 else 0.0, slope=np.polyfit(t, w, 1)[0],
                zcr=np.mean(np.diff(np.sign(x)) != 0), ac1=ac1 if np.isfinite(ac1) else 0.0)

def window_table(d, cols, L=15, hop=15, label="occupancy", names=None):
    """Taie un DataFrame indexat în timp în ferestre de L eșantioane, la fiecare hop eșantioane.
    Returnează X (un rând pe fereastră), y (media lui `label` în fereastră), day (grupul = ziua săptămânii)."""
    names = names or SHORT; rows, ys, days, t0 = [], [], [], []
    A = {c: d[c].values for c in cols}; lab = d[label].values
    for s in range(0, len(d) - L + 1, hop):
        r = {}
        for c in cols:
            for k, v in stat_features(A[c][s:s + L]).items(): r[f"{names.get(c, c)}_{k}"] = v
        rows.append(r); ys.append(float(np.mean(lab[s:s + L]))); days.append(d.index[s].dayofweek); t0.append(d.index[s])
    return pd.DataFrame(rows, index=t0).fillna(0.0), np.array(ys), np.array(days)

def make_bearing(n_per=150, seed=0):
    """Simulator de vibrații: eticheta 0 = sănătos, 1 = defect de rulment. Returnează X (n_ferestre x 2000 eșantioane), y. fs = 2000 Hz."""
    r = np.random.default_rng(seed); t = np.arange(2000) / FS_B; X, y = [], []
    for lab in (0, 1):
        for _ in range(n_per):
            f0 = r.uniform(29, 31); a = r.uniform(0.8, 1.2)
            x = a * np.sin(2 * np.pi * f0 * t + r.uniform(0, 6.28)) + 0.4 * a * np.sin(2 * np.pi * 2 * f0 * t + r.uniform(0, 6.28))
            x += r.normal(0, r.uniform(0.35, 0.55), 2000)
            if lab == 1:
                bp = r.uniform(84, 90); amp = r.uniform(0.9, 1.8); fr = r.uniform(560, 640)
                k = np.arange(0, 0.012, 1 / FS_B); imp = np.exp(-k / 0.004) * np.sin(2 * np.pi * fr * k)
                imps = np.zeros(2000)
                for e in np.arange(r.uniform(0, 1 / bp), 1, 1 / bp):
                    i = int(e * FS_B); m = min(len(imp), 2000 - i)
                    if m > 0: imps[i:i + m] += imp[:m] * amp * r.uniform(0.7, 1.3)
                x = x + imps; x *= r.uniform(0.80, 0.92)
            X.append(x); y.append(lab)
    return np.array(X), np.array(y)

def spec_features(x, fs=FS_B, bands=((0, 100), (100, 300), (300, 500), (500, 800), (800, 1000))):
    """Puterile pe benzi (Welch) + centroidul spectral, frecvența dominantă și planeitatea unei ferestre."""
    f, P = signal.welch(x, fs, nperseg=512, window="hann"); df = f[1] - f[0]; d = {}
    for lo, hi in bands: d[f"bp_{lo}_{hi}"] = P[(f >= lo) & (f < hi)].sum() * df
    pn = P / P.sum(); d["centroid"] = (f * pn).sum(); d["domf"] = f[np.argmax(P)]
    d["flat"] = np.exp(np.mean(np.log(P + 1e-12))) / np.mean(P)
    return d

def wavelet_energy(x, wavelet="db4", level=5):
    c = pywt.wavedec(x, wavelet, level=level); E = np.array([np.sum(ci ** 2) for ci in c]); E = E / E.sum()
    return {f"wE_{i}": E[i] for i in range(len(E))}

def time_features(x):
    x = np.asarray(x, float); rms = np.sqrt(np.mean(x ** 2))
    return dict(rms=rms, std=x.std(), kurt=stats.kurtosis(x), crest=np.max(np.abs(x)) / rms, pp=np.ptp(x),
                zcr=np.mean(np.diff(np.sign(x - x.mean())) != 0))

def bearing_table(X):
    rows = []
    for x in X:
        r = {}; r.update(time_features(x)); r.update(spec_features(x)); r.update(wavelet_energy(x)); rows.append(r)
    return pd.DataFrame(rows)

def load_occupancy(path="Occupancy_Estimation.csv", rule="5min"):
    """Date reale UCI de ocupare a unei camere (7 noduri senzor, 30 s) -> un rând pe fereastră de 5 minute; day = grupul, count = media numărului de persoane."""
    d = pd.read_csv(path); d["ts"] = pd.to_datetime(d.Date + " " + d.Time); d = d.set_index("ts").drop(columns=["Date", "Time"])
    d["Temp"] = d[[f"S{i}_Temp" for i in range(1, 5)]].mean(axis=1); d["Light"] = d[[f"S{i}_Light" for i in range(1, 5)]].mean(axis=1)
    d["Sound"] = d[[f"S{i}_Sound" for i in range(1, 5)]].mean(axis=1); d["PIR"] = d[["S6_PIR", "S7_PIR"]].max(axis=1)
    r = d.resample(rule); g = r[["Temp", "Light", "Sound", "S5_CO2", "S5_CO2_Slope", "PIR", "Room_Occupancy_Count"]].agg(["mean", "std", "min", "max"])
    g.columns = ["_".join(c) for c in g.columns]; g = g[(r.size() >= 8).values].dropna()
    cnt = g["Room_Occupancy_Count_mean"].values; X = g.drop(columns=[c for c in g if c.startswith("Room")])
    return X, cnt, np.array([t.dayofyear for t in g.index])

def make_mix(seed, fs=500, seconds=4):
    "două tonuri ascunse (Hz întregi) + zgomot. returnează t, x, fs"
    rng = np.random.default_rng(seed); t = np.arange(0, seconds, 1 / fs)
    f1 = int(rng.integers(20, 60)); f2 = int(rng.integers(90, 140))
    return t, 3 * np.sin(2 * np.pi * f1 * t) + 1.5 * np.sin(2 * np.pi * f2 * t + 1) + 0.5 * rng.normal(size=t.size), fs

def make_blocks(seed, n_blocks=24, B=600, Lw=60, hop=6):
    "blocuri de zgomot autocorelat cu etichetă ALEATOARE pe bloc, tăiate în ferestre puternic suprapuse. returnează X, y, g"
    rng = np.random.default_rng(seed); lab = rng.integers(0, 2, n_blocks); feats, ys, grp = [], [], []
    for b in range(n_blocks):
        e = rng.normal(size=B); ar = np.zeros(B)
        for i in range(1, B): ar[i] = 0.9 * ar[i - 1] + e[i]
        s = ar + rng.normal()
        for st in range(0, B - Lw + 1, hop): feats.append(stat_features(s[st:st + Lw])); ys.append(lab[b]); grp.append(b)
    return pd.DataFrame(feats), np.array(ys), np.array(grp)

def make_noise_task(seed, n=100, p=500):
    "etichetele sunt aleatoare: nicio trăsătură nu poartă informație"
    rng = np.random.default_rng(seed); return rng.normal(size=(n, p)), rng.integers(0, 2, n)

# Partea 1 — De la semnal la un tabel de trăsături

Funcții deja definite pentru voi: `load_office()`, `stat_features(w)`, `window_table(d, cols, L, hop, label)`, `load_occupancy(rule)`, `make_blocks(seed)`. Folosiți-le ca în exemplele rezolvate.

## 1.1 · Câte ferestre?  ★☆☆

Înregistrarea biroului are `N = 10080` eșantioane (unul pe minut).

* `n_15_5` – numărul de ferestre complete cu `L = 15`, `hop = 5`;
* `n_60_30` – cu `L = 60`, `hop = 30`;
* `L_for_100` – lungimea ferestrei (în minute, **fără suprapunere**) care dă aproximativ **100** de ferestre (rotunjiți `N / 100` în jos la un *L* întreg și verificați că obțineți ≥ 100).

**Gândiți-vă:** dacă dublați suprapunerea, obțineți de două ori mai multe exemple *independente*?

In [ ]:
d = load_office(); N = len(d)
n_15_5 = ...
n_60_30 = ...
L_for_100 = ...

**Autoverificare**

In [ ]:
check_value("n_15_5", n_15_5, "4014f8c0", "floor((N - L) / hop) + 1")
check_value("n_60_30", n_60_30, "b1e68467", "L = 60, hop = 30")
check_value("L_for_100", L_for_100, "d66c66a9", "ferestre f\u0103r\u0103 suprapunere: N / L = 100 -> L = ?")

**Interpretarea voastră:** _scrieți aici_

## 1.2 · Regula de etichetare contează  ★★☆

Tăiați datele biroului în **ferestre de 30 de minute fără suprapunere**: `X, yp, day = window_table(d, cols, 30, 30)` (`yp` = numărul mediu de persoane din fiecare fereastră), cu `cols = ["temp_c", "humidity_pct", "co2_ppm"]`.

* `n_gt05` – numărul de ferestre cu `yp > 0.5` (regula noastră);
* `n_all` – numărul de ferestre în care **fiecare** minut este ocupat (regula strictă *„ocupat tot timpul”*);
* `n_straddle` – numărul de ferestre care conțin **și** minute goale, **și** ocupate (folosiți `d.occupancy` brut, remodelat la `(-1, 30)`).

**Gândiți-vă:** ați alege regula strictă sau regula majorității pentru un controler de ventilație și ce s-ar schimba pentru o alarmă de securitate? De ce trebuie regula scrisă în raport?

In [ ]:
d = load_office()
cols = ["temp_c", "humidity_pct", "co2_ppm"]
X, yp, day = window_table(d, cols, 30, 30)
n_gt05 = ...
n_all = ...
n_straddle = ...

**Autoverificare**

In [ ]:
check_value("n_gt05", n_gt05, "37a1151e", "ferestre a c\u0103ror medie de persoane > 0,5")
check_value("n_all", n_all, "a8a9a446", "ferestre \u00een care FIECARE minut este ocupat")
check_value("n_straddle", n_straddle, "9ae1524b", "ferestre cu minute \u0219i goale, \u0219i ocupate")

**Interpretarea voastră:** _scrieți aici_

## 1.3 · Alegerea lungimii ferestrei  ★★☆

Pentru `L` în `{10, 20, 30}` (fără suprapunere), construiți tabelul cu `window_table(d, cols, L, L)`, eticheta `(yp > 0.5)` și evaluați **StandardScaler + LogisticRegression(max_iter=3000)** cu **lasă-o-zi-afară** (`LeaveOneGroupOut`, `groups=day`). Memorați **acuratețea echilibrată** a fiecărei lungimi în `bacc_10`, `bacc_20`, `bacc_30`.

**Scrieți:** ce v-ar face să alegeți 10 minute în loc de 30, deși scorul este puțin mai mic?

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import balanced_accuracy_score as bacc
d = load_office()
cols = ["temp_c", "humidity_pct", "co2_ppm"]
scores = {}
for L in (10, 20, 30):
    ...
bacc_10, bacc_20, bacc_30 = scores[10], scores[20], scores[30]

**Autoverificare**

In [ ]:
check_value("bacc_10", bacc_10, "b8cbc389", "L = 10")
check_value("bacc_20", bacc_20, "b8cbc389", "L = 20")
check_value("bacc_30", bacc_30, "b45898ec", "L = 30")
check("scorurile sunt într-un interval rezonabil (0,9 - 1,0)", all(0.9 <= v <= 1.0 for v in (bacc_10, bacc_20, bacc_30)), "un pipeline StandardScaler + LogisticRegression, lasă-o-zi-afară")

**Interpretarea voastră:** _scrieți aici_

## 1.4 · Datele reale la altă lungime de fereastră  ★★☆

`load_occupancy(rule)` returnează `X, cnt, dayo`: tabelul de ferestre, numărul mediu de persoane pe fereastră și ziua fiecărei ferestre. Implicit `rule="5min"`. Apelați-o cu `rule="10min"` și raportați:

* `n_windows` – numărul de rânduri ale tabelului;
* `n_days` – numărul de zile diferite (grupuri);
* `share_occ` – fracția de ferestre cu `cnt > 0.5` (nu este nevoie de rotunjire la 3 zecimale).

**Gândiți-vă:** de ce numărul de ferestre nu se înjumătățește exact când lungimea ferestrei se dublează?

In [ ]:
X10, cnt10, day10 = load_occupancy(rule="10min")
n_windows = ...
n_days = ...
share_occ = ...

**Autoverificare**

In [ ]:
check_value("n_windows", n_windows, "c5a0b258", "r\u00e2ndurile tabelului la 10 minute")
check_value("n_days", n_days, "b2337ecb", "num\u0103rul de grupuri diferite")
check_value("share_occ", share_occ, "38628535", "ponderea ferestrelor cu count > 0,5")

**Interpretarea voastră:** _scrieți aici_

## 1.5 · Demonstrați scurgerea de informație  ★★★

`Xn, yn, g = make_blocks(7)` produce ferestre de zgomot autocorelat ale căror etichete sunt **aleatorii** pe bloc (`g` = id-ul blocului): nu se poate învăța nimic, iar acuratețea echilibrată cinstită este ≈ 0,5.

Evaluați un `DecisionTreeClassifier(random_state=0)` cu

* `acc_random` – `KFold(5, shuffle=True, random_state=0)`;
* `acc_grouped` – `LeaveOneGroupOut()` cu `groups=g`;

și setați `n_blocks` la numărul de blocuri. **Scrieți:** explicați în două propoziții de ce primul număr este mare.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import KFold, LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import balanced_accuracy_score as bacc
Xn, yn, g = make_blocks(7)
acc_random = ...
acc_grouped = ...
n_blocks = ...

**Autoverificare**

In [ ]:
check("împărțirea aleatorie pare bună (> 0,65)", acc_random > 0.65, "ferestrele suprapuse „scurg” informație între antrenare și test")
check("împărțirea pe grupuri spune adevărul (< 0,60)", acc_grouped < 0.60, "grup = bloc (`g`)")
check_value("n_blocks", n_blocks, "85d40e1b", "num\u0103rul de grupuri")

**Interpretarea voastră:** _scrieți aici_

# Partea 2 — Trei familii de trăsături

Funcții ajutătoare: `time_features(x)`, `spec_features(x)`, `wavelet_energy(x)`, `bearing_table(X)`, `make_bearing(n_per, seed)`, `make_mix(seed)`; frecvența de eșantionare a rulmentului este `FS_B = 2000` Hz.

## 2.1 · Trăsăturile unor semnale cunoscute  ★☆☆

Construiți o **sinusoidă de amplitudine 3 și frecvență 40 Hz**, eșantionată la 1000 Hz timp de 1 s, și zgomot gaussian `np.random.default_rng(1).normal(size=1000)`. Cu `time_features`:

* `rms`, `kurt`, `crest` ale sinusoidei; `kurt_noise` – kurtosisul zgomotului.

**Verificați-vă intuiția:** ar trebui să puteți prezice primele trei numere fără calculator (A/√2, −1,5, √2).

In [ ]:
t = np.arange(1000) / 1000
sine = ...
noise = np.random.default_rng(1).normal(size=1000)
f_s = time_features(sine)
rms, kurt, crest = ...
kurt_noise = ...

**Autoverificare**

In [ ]:
check_value("rms", rms, "f487eb52", "A / sqrt(2)")
check_value("kurt", kurt, "31041bdd", "o sinusoid\u0103 pur\u0103")
check_value("crest", crest, "9909ad1a", "max|x| / rms")
check_value("kurt_noise", kurt_noise, "e6743df8", "zgomot gaussian")

**Interpretarea voastră:** _scrieți aici_

## 2.2 · Găsiți tonurile ascunse  ★★☆

`t, x, fs = make_mix(21)` conține **două sinusoide** cu frecvențe întregi ascunse, plus zgomot. Folosiți `np.fft.rfft` și `np.fft.rfftfreq`:

* `f_low`, `f_high` – frecvențele (Hz) celor două vârfuri spectrale cele mai mari (întâi cea mai joasă);
* `df` – rezoluția în frecvență, în Hz (distanța dintre canalele FFT).

**Gândiți-vă:** semnalul durează 4 s. Cât ar trebui să înregistrați ca să separați două tonuri aflate la 0,1 Hz distanță?

In [ ]:
t, x, fs = make_mix(21)
spec = np.abs(np.fft.rfft(x)); freq = np.fft.rfftfreq(len(x), 1 / fs)
f_low, f_high = ...
df = ...

**Autoverificare**

In [ ]:
check_value("f_low", f_low, "465ae46c", "cea mai joas\u0103 dintre cele dou\u0103 tonuri (Hz)")
check_value("f_high", f_high, "93a4d9ad", "tonul mai \u00eenalt")
check_value("df", df, "a30a0433", "fs / N")

**Interpretarea voastră:** _scrieți aici_

## 2.3 · Aliasing  ★★☆

Un ton de 130 Hz este eșantionat la doar `fs = 200` Hz timp de 2 s.

* `apparent_fft` – frecvența celui mai mare vârf FFT pe care îl observați efectiv;
* `apparent` – aceeași valoare prezisă de teorie (tonul se împăturește în jurul lui `fs/2`: `|130 − 200|`);
* `min_fs` – cea mai mică frecvență de eșantionare (Hz) care respectă Nyquist pentru acest ton.

**Gândiți-vă:** datele arată un ton curat de 70 Hz. Ați putea ști, numai din înregistrarea la 200 Hz, că frecvența reală era 130 Hz?

In [ ]:
fs = 200; t = np.arange(0, 2, 1 / fs); x = np.sin(2 * np.pi * 130 * t)
spec = np.abs(np.fft.rfft(x)); freq = np.fft.rfftfreq(len(x), 1 / fs)
apparent_fft = ...
apparent = ...
min_fs = ...

**Autoverificare**

In [ ]:
check_value("apparent", apparent, "e60d2ea7", "|f - k * fs| \u00eemp\u0103turit sub fs/2")
check_value("min_fs", min_fs, "3f6ae649", "2 x cea mai mare frecven\u021b\u0103")
check("ați calculat frecvența aparentă din FFT", abs(apparent_fft - apparent) < 1, "np.fft.rfftfreq(...)[argmax]")

**Interpretarea voastră:** _scrieți aici_

## 2.4 · Ce bandă poartă defectul?  ★★☆

`Xb, yb = make_bearing(100, 5)`. Pentru fiecare bandă din `bands = [(0,100), (100,300), (300,500), (500,800), (800,1000)]` calculați puterea Welch pe bandă a fiecărei ferestre (`spec_features` le returnează ca `bp_lo_hi`; folosiți `bearing_table(Xb)`).

* `best_band` – șirul `"lo_hi"` al benzii (de exemplu `"100_300"`) pentru care **puterea medie în ferestrele defecte ÷ puterea medie în ferestrele sănătoase** este cea mai mare;
* `ratio` – acel raport.

**Scrieți:** cărei componente fizice a rulmentului îi corespunde această bandă?

In [ ]:
Xb, yb = make_bearing(100, 5); F = bearing_table(Xb)
bands = [(0, 100), (100, 300), (300, 500), (500, 800), (800, 1000)]
best_band = ...
ratio = ...

**Autoverificare**

In [ ]:
check_value("best_band", best_band, "c37cb13b", "banda cu cel mai mare raport defect / s\u0103n\u0103tos al puterii medii pe band\u0103")
check_value("ratio", ratio, "980d4a2a", "puterea medie a ferestrelor defecte / puterea medie a celor s\u0103n\u0103toase \u00een acea band\u0103")
check("ratio > 1.5", ratio > 1.5, "")

**Interpretarea voastră:** _scrieți aici_

## 2.5 · Ce nivel wavelet?  ★★☆

Pentru `fs = 2000`, `db4`, 5 niveluri, `pywt.wavedec` returnează `[cA5, cD5, cD4, cD3, cD2, cD1]` (indici 0…5).

Generați sinusoide de 1 s la **200 Hz** și la **700 Hz**; calculați energia fiecărui vector de coeficienți (`np.sum(c**2)`) și dați `level` = indexul energiei maxime pentru 200 Hz și `level_700` pentru 700 Hz.

**Gândiți-vă:** preziceți mai întâi ambii indici din tabelul `fs/2^(j+1) … fs/2^j`.

In [ ]:
import pywt
t = np.arange(2000) / 2000
def top_level(f):
    ...
level = top_level(200)
level_700 = top_level(700)

**Autoverificare**

In [ ]:
check_value("level", level, "a97491cd", "indexul \u00een [cA5, cD5, cD4, cD3, cD2, cD1] = 0..5 al celei mai mari energii")
check_value("level_700", level_700, "9ae1524b", "la fel pentru un ton de 700 Hz")
check("cele două tonuri cad în niveluri diferite", level != level_700, "")

**Interpretarea voastră:** _scrieți aici_

## 2.6 · Comparați familiile de trăsături  ★★★

Cu `Xb, yb = make_bearing(150, 3)` și `F = bearing_table(Xb)`, evaluați **StandardScaler + LogisticRegression(max_iter=3000)** cu `StratifiedKFold(5, shuffle=True, random_state=0)`, folosind acuratețea echilibrată a lui `cross_val_predict`, pentru trei seturi de trăsături:

* `s_rms` – `["rms"]`; `s_kurt_crest` – `["kurt", "crest"]`; `s_bands` – cele cinci coloane care încep cu `bp_`.
* `best_family` – **șirul** `"rms"`, `"kurt_crest"` sau `"bands"`, care a obținut scorul cel mai mare.

**Scrieți:** de ce familia cu acuratețea cea mai mare este și cea mai bine justificată de fizică?

In [ ]:
Xb, yb = make_bearing(150, 3); F = bearing_table(Xb)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
def score(cols):
    ...
s_rms = score(["rms"]); s_kurt_crest = score(["kurt", "crest"]); s_bands = score([c for c in F if c.startswith("bp_")])
best_family = ...

**Autoverificare**

In [ ]:
check_value("s_rms", s_rms, "90e68311", "doar RMS")
check_value("s_kurt_crest", s_kurt_crest, "4d44dd7e", "kurtosis + factor de v\u00e2rf")
check_value("s_bands", s_bands, "b45898ec", "cele 5 puteri pe band\u0103")
check_value("best_family", best_family, "1a5db2ab", "numele familiei cu cel mai mare scor")

**Interpretarea voastră:** _scrieți aici_

## 2.7 · Ritmul biroului  ★★☆

Luați CO₂ curățat pe toată săptămâna (`load_office().co2_ppm`, un eșantion pe minut), **scădeți media**, calculați FFT și găsiți **perioada dominantă** în ore.

* `period_h` – perioada (în ore) a celui mai mare vârf spectral (excludeți frecvența zero).

**Gândiți-vă:** ce proces fizic creează această perioadă și de ce o fereastră de 15 minute nu ar vedea-o niciodată?

In [ ]:
co2 = load_office().co2_ppm.values
period_h = ...

**Autoverificare**

In [ ]:
check_value("period_h", period_h, "85d40e1b", "1 / frecven\u021b\u0103, \u00een ore")
check("un ciclu zilnic este cel dominant", abs(period_h - 24) < 2, "înregistrarea CO2 durează 7 zile")

**Interpretarea voastră:** _scrieți aici_

## 2.8 · Spectrul unui senzor de lumină  ★★☆

*(din laboratorul inițial: „aplicați transformata Fourier pe senzorul Light”)* Pe datele reale de ocupare, `Xo, cnt, dayo = load_occupancy()` dă ferestre de 5 minute; luați **cea mai lungă zi** și coloana ei `Light_mean` (lumina medie a celor patru noduri). Scădeți media, calculați spectrul de putere `|rfft|²` și frecvențele în **cicluri pe oră** (`np.fft.rfftfreq(len(x), d=5/60)`):

* `period_dom_min` – perioada (în minute) a celui mai mare vârf (excludeți f = 0);
* `share_low` – ponderea din puterea totală (f > 0) aflată sub **0,5 cicluri/oră** (perioade mai lungi de 2 h);
* `peak_ratio` – cea mai mare putere împărțită la puterea **mediană** (f > 0): un semnal periodic are un raport mare.

**Scrieți:** vedeți un tipar recurent al iluminării? Ce înseamnă când „perioada” dominantă este egală cu întreaga înregistrare? De ce o zi înregistrată doar ~10 ore este o bază slabă pentru o analiză a ciclului zilnic?

In [ ]:
Xo, cnt, dayo = load_occupancy()
days = np.unique(dayo); big = days[np.argmax([(dayo == k).sum() for k in days])]       # cea mai lungă zi
light = Xo["Light_mean"].values[dayo == big]
x = light - light.mean()
spec = np.abs(np.fft.rfft(x)) ** 2; freq = np.fft.rfftfreq(len(x), d=5 / 60)           # cicluri pe oră
period_dom_min = ...
share_low = ...
peak_ratio = ...

**Autoverificare**

In [ ]:
check_value("period_dom_min", period_dom_min, "e2dc2b69", "1 / frecven\u021ba dominant\u0103, \u00een minute (f > 0)")
check_value("share_low", share_low, "119e73ca", "puterea la f < 0,5 cicluri/or\u0103 \u00eemp\u0103r\u021bit\u0103 la puterea total\u0103 (f > 0)")
check_value("peak_ratio", peak_ratio, "5194a2a4", "putere maxim\u0103 / putere median\u0103, f > 0")
check("„perioada” dominantă este doar lungimea înregistrării (o tendință, nu un ciclu)", period_dom_min > 0.8 * len(x) * 5, "comparați cu len(x) * 5 minute")

**Interpretarea voastră:** _scrieți aici_

## 2.9 · Tipul și nivelul wavelet pe senzorul Sound  ★★☆

*(din laboratorul inițial: „experimentați cu tipuri și niveluri de wavelet pe senzorul Sound”)* Luați primele 2048 de eșantioane din `S1_Sound` din `Occupancy_Estimation.csv` (eșantioane la 30 de secunde, ≈ 17 ore), scădeți media și descompuneți cu `pywt.wavedec` la **nivelul 3**:

* `len_haar_L3`, `len_sym5_L3` – lungimea aproximării `cA3` pentru `haar` și pentru `sym5`;
* `frac_haar_L3`, `frac_sym5_L3` – energia lui `cA3` împărțită la energia totală a tuturor coeficienților;
* `max_level_sym5` – cel mai adânc nivel util pentru această lungime a semnalului cu `sym5` (`pywt.dwt_max_level`).

**Gândiți-vă:** de ce cele două lungimi sunt diferite? Ce vă spune fracția de energie a lui `cA3` despre cât de *lent* este semnalul Sound? Dacă ați vrea o trăsătură care descrie componenta lentă, ce nivel ați alege și cum schimbă alegerea wavelet-ului rezultatul?

In [ ]:
import pywt
raw = pd.read_csv("Occupancy_Estimation.csv")
s = raw["S1_Sound"].values[:2048].astype(float); s = s - s.mean()
def decompose(wavelet, level=3):
    c = pywt.wavedec(s, wavelet, level=level); E = np.array([np.sum(ci ** 2) for ci in c])
    return len(c[0]), E[0] / E.sum()
len_haar_L3, frac_haar_L3 = ...
len_sym5_L3, frac_sym5_L3 = ...
max_level_sym5 = ...

**Autoverificare**

In [ ]:
check_value("len_haar_L3", len_haar_L3, "066621a8", "lungimea aproxim\u0103rii cA3 cu haar")
check_value("len_sym5_L3", len_sym5_L3, "743973c2", "filtrele mai lungi produc c\u00e2\u021biva coeficien\u021bi \u00een plus la margini")
check_value("frac_haar_L3", frac_haar_L3, "ad9dadb8", "energia lui cA3 / energia total\u0103 (dup\u0103 sc\u0103derea mediei)")
check_value("frac_sym5_L3", frac_sym5_L3, "3eb631bf", "la fel pentru sym5")
check_value("max_level_sym5", max_level_sym5, "b2337ecb", "pywt.dwt_max_level(len(s), \"sym5\")")

**Interpretarea voastră:** _scrieți aici_

# Partea 3 — Selecția trăsăturilor

Toate sarcinile acestei părți folosesc tabelul de ferestre al biroului (ferestre de 15 minute fără suprapunere, 36 de trăsături), construit pentru voi în fiecare sarcină cu `window_table` (exercițiile 3.7–3.9 folosesc în schimb tabelul real de ocupare). Rețineți regula de aur: **selecția trăiește în interiorul pipeline-ului.**

## 3.1 · Ordonați trăsăturile  ★☆☆

* `n_high` – câte trăsături au o corelație Pearson absolută cu `y` **peste 0,8**;
* `top_mi` – **numele** trăsăturii cu cea mai mare informație mutuală (`mutual_info_classif(X, y, random_state=0)`);
* `n_hum_mid` – câte dintre trăsăturile de umiditate (nume care încep cu `hum_`) au corelația absolută peste 0,5.

**Gândiți-vă:** trăsăturile din frunte după informația mutuală sunt aproape copii una alteia. De ce contează?

In [ ]:
d = load_office(); cols = ["temp_c", "humidity_pct", "co2_ppm"]
X, yp, day = window_table(d, cols, 15, 15); y = (yp > 0.5).astype(int)

corr_y = ...
n_high = ...
top_mi = ...
n_hum_mid = ...

**Autoverificare**

In [ ]:
check_value("n_high", n_high, "a8994a90", "|corr| > 0,8")
check_value("top_mi", top_mi, "fd999533", "numele tr\u0103s\u0103turii cu cea mai mare informa\u021bie mutual\u0103 (random_state=0)")
check_value("n_hum_mid", n_hum_mid, "9ae1524b", "tr\u0103s\u0103turi de umiditate (nume care \u00eencep cu \"hum_\") cu |corr| > 0,5")

**Interpretarea voastră:** _scrieți aici_

## 3.2 · Eliminați redundanța  ★★☆

* `n_pairs` – numărul de perechi de trăsături (i < j) cu **|r| > 0,95**;
* `n_after` – numărul de trăsături păstrate de **regula lacomă**: parcurgeți coloanele în ordinea dată și **păstrați** o coloană doar dacă |r| cu fiecare coloană deja păstrată este ≤ 0,95.

**Gândiți-vă:** ce ați pierde dacă ați folosi *corelația cu ținta* pentru a decide care dintre două copii să rămână?

In [ ]:
d = load_office(); cols = ["temp_c", "humidity_pct", "co2_ppm"]
X, yp, day = window_table(d, cols, 15, 15); y = (yp > 0.5).astype(int)

C = X.corr().abs()
n_pairs = ...
keep = []
for c in X.columns:
    ...
n_after = len(keep)

**Autoverificare**

In [ ]:
check_value("n_pairs", n_pairs, "6a33755d", "perechi (i < j) cu |r| > 0,95")
check_value("n_after", n_after, "85d40e1b", "coloane p\u0103strate de regula lacom\u0103")
check("rămân mai puține coloane, dar nu prea puține (15 până la 30)", 15 <= n_after <= 30, "")

**Interpretarea voastră:** _scrieți aici_

## 3.3 · RFE în interiorul pipeline-ului  ★★☆

Folosind un pipeline **StandardScaler → RFE(LogisticRegression(max_iter=3000), n_features_to_select=5) → LogisticRegression(max_iter=3000)**:

* `score_rfe` – acuratețea echilibrată pe date nevăzute cu lasă-o-zi-afară;
* `score_all` – la fel, cu toate cele 36 de trăsături și fără selecție;
* `n_kept_slope` – antrenați RFE pe **tabelul complet standardizat** și numărați câte trăsături selectate se termină în `_slope`.

In [ ]:
d = load_office(); cols = ["temp_c", "humidity_pct", "co2_ppm"]
X, yp, day = window_table(d, cols, 15, 15); y = (yp > 0.5).astype(int)

LR = lambda: LogisticRegression(max_iter=3000)
score_rfe = ...
score_all = ...
n_kept_slope = ...

**Autoverificare**

In [ ]:
check_value("n_kept_slope", n_kept_slope, "cf9dcf6d", "c\u00e2te dintre cele 5 tr\u0103s\u0103turi RFE se termin\u0103 \u00een \"_slope\"")
check_value("score_rfe", score_rfe, "bee67f04", "acurate\u021be echilibrat\u0103, las\u0103-o-zi-afar\u0103, selec\u021bie \u00een interiorul pipeline-ului")
check("aproape de toate trăsăturile (la cel mult 0,03)", abs(score_rfe - score_all) < 0.03, "")

**Interpretarea voastră:** _scrieți aici_

## 3.4 · Penalizarea L1 ca selector  ★★☆

Antrenați `LogisticRegression(penalty="l1", solver="liblinear", C=C, random_state=0)` pe tabelul **standardizat** pentru `C = 0.01, 0.1, 1` și numărați **ponderile nenule**: `nz_001`, `nz_01`, `nz_1`.

**Gândiți-vă:** ce `C` ați folosi dacă ați avea nevoie de un model cu cel mult 5 intrări? Ce riscați mergând mai jos?

In [ ]:
d = load_office(); cols = ["temp_c", "humidity_pct", "co2_ppm"]
X, yp, day = window_table(d, cols, 15, 15); y = (yp > 0.5).astype(int)

Z = StandardScaler().fit_transform(X)
def n_nonzero(C):
    ...
nz_001, nz_01, nz_1 = n_nonzero(0.01), n_nonzero(0.1), n_nonzero(1)

**Autoverificare**

In [ ]:
check_value("nz_001", nz_001, "9ae1524b", "C = 0,01")
check_value("nz_01", nz_01, "9ae1524b", "C = 0,1")
check_value("nz_1", nz_1, "df644619", "C = 1")
check("mai multe trăsături supraviețuiesc pe măsură ce C crește", nz_001 <= nz_01 <= nz_1, "")

**Interpretarea voastră:** _scrieți aici_

## 3.5 · Scurgere de informație prin selecție  ★★★

`Xn, yn = make_noise_task(11)` are 100 de exemple, 500 de trăsături și **etichete aleatorii**. Cu `StratifiedKFold(5, shuffle=True, random_state=0)` și `LogisticRegression(max_iter=3000)`:

* `wrong` – mai întâi alegeți cele 10 trăsături cele mai bune cu `SelectKBest(f_classif, k=10)` pe **toate** datele, apoi validați încrucișat (acuratețea echilibrată medie);
* `right` – puneți `SelectKBest` **în interiorul** lui `make_pipeline` și validați încrucișat;
* `gap_big` – `True` dacă `wrong - right > 0.1`.

**Scrieți:** ce regulă din Cursul 6 demonstrează acest lucru?

In [ ]:
Xn, yn = make_noise_task(11)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
LR = LogisticRegression(max_iter=3000)
wrong = ...
right = ...
gap_big = ...

**Autoverificare**

In [ ]:
check("selecția în afara CV arată bine (> 0,62)", wrong > 0.62, "a văzut toate etichetele")
check("selecția în interiorul pipeline-ului este cinstită (< 0,58)", right < 0.58, "SelectKBest în interiorul make_pipeline")
check_value("gap_big", gap_big, "3cbc87c7", "este wrong - right mai mare dec\u00e2t 0,1 ? (True / False)")

**Interpretarea voastră:** _scrieți aici_

## 3.6 · Cât de stabilă este selecția?  ★★★

Pentru fiecare dintre cele 7 seturi de antrenare lasă-o-zi-afară, standardizați (calibrat doar pe zilele de antrenare) și antrenați regresia logistică L1 (`solver="liblinear"`, `C=0.1`, `random_state=0`). Numărați în câte fold-uri fiecare trăsătură are pondere nenulă.

* `n_stable` – numărul de trăsături păstrate în **toate cele 7** fold-uri.

**Gândiți-vă:** pe care dintre trăsăturile stabile le-ați păstra în modelul final și de ce nu ați interpreta ponderea uneia instabile?

In [ ]:
d = load_office(); cols = ["temp_c", "humidity_pct", "co2_ppm"]
X, yp, day = window_table(d, cols, 15, 15); y = (yp > 0.5).astype(int)

count = pd.Series(0, index=X.columns)
for tr, te in LeaveOneGroupOut().split(X, y, day):
    ...
n_stable = ...

**Autoverificare**

In [ ]:
check_value("n_stable", n_stable, "8c9d9d6d", "p\u0103strate \u00een toate cele 7 fold-uri")
check("cel puțin o trăsătură este stabilă", n_stable >= 1, "")

**Interpretarea voastră:** _scrieți aici_

## 3.7 · Analiza corelației pe datele reale  ★★☆

*(din laboratorul inițial, Exercițiul 1 din secțiunea 2.5)* Pe tabelul **real** de ocupare (24 de trăsături, ferestre de 5 minute):

* `n_pairs90` – numărul de perechi de trăsături cu corelație absolută **peste 0,90**;
* `n_kept` – numărul de trăsături păstrate de regula lacomă (parcurgeți coloanele în ordine; păstrați o coloană doar dacă |r| cu fiecare coloană păstrată este ≤ 0,90);
* `bacc_full`, `bacc_reduced` – acuratețea echilibrată pe date nevăzute (lasă-o-zi-afară) a **StandardScaler + LogisticRegression** cu toate cele 24 de trăsături și cu cele păstrate.

**Scrieți:** setul redus este mai mic — este modelul mai slab? Pe care dintre trăsăturile eliminate v-ați fi părut rău să le pierdeți și de ce?

In [ ]:
Xo, cnt, dayo = load_occupancy(); yo = (cnt > 0.5).astype(int)
lodo = LeaveOneGroupOut(); LR = lambda: LogisticRegression(max_iter=3000)
f = lambda m, cols: bacc(yo, cross_val_predict(m, Xo[cols], yo, cv=lodo, groups=dayo))

C = Xo.corr().abs()
n_pairs90 = ...
keep = []
for c in Xo.columns:
    ...
n_kept = len(keep)
bacc_full = ...
bacc_reduced = ...

**Autoverificare**

In [ ]:
check_value("n_pairs90", n_pairs90, "3d03d318", "perechi (i < j) cu |r| > 0,90")
check_value("n_kept", n_kept, "0856fec2", "regula lacom\u0103 cu pragul 0,90, coloane \u00een ordinea dat\u0103")
check_value("bacc_full", bacc_full, "b8cbc389", "toate cele 24 de tr\u0103s\u0103turi, las\u0103-o-zi-afar\u0103")
check_value("bacc_reduced", bacc_reduced, "b45898ec", "doar tr\u0103s\u0103turile p\u0103strate")

**Interpretarea voastră:** _scrieți aici_

## 3.8 · RFE: primele 3 trăsături, cinstit și necinstit  ★★★

*(din laboratorul inițial, Exercițiul 2 din secțiunea 2.5)* Folosiți toate cele 24 de trăsături ale tabelului real și regresia logistică:

* `rfe_features` – cele trei trăsături alese de `RFE(LR(), n_features_to_select=3)` antrenat pe **tabelul complet standardizat**; dați numele sortate alfabetic și unite cu `","`;
* `bacc_rfe_inside` – acuratețea echilibrată pe date nevăzute (lasă-o-zi-afară) a pipeline-ului **StandardScaler → RFE(3) → LogisticRegression**;
* `bacc_rfe_outside` – acuratețea echilibrată a **StandardScaler + LogisticRegression** care folosește doar cele trei trăsături de mai sus (alese **o singură dată, pe toate zilele**).

**Scrieți:** care număr este cinstit? Explicați diferența în două propoziții și comparați cu setul complet (3.7).

In [ ]:
Xo, cnt, dayo = load_occupancy(); yo = (cnt > 0.5).astype(int)
lodo = LeaveOneGroupOut(); LR = lambda: LogisticRegression(max_iter=3000)
f = lambda m, cols: bacc(yo, cross_val_predict(m, Xo[cols], yo, cv=lodo, groups=dayo))

Z = StandardScaler().fit_transform(Xo)
rfe_features = ...
bacc_rfe_inside = ...
bacc_rfe_outside = ...

**Autoverificare**

In [ ]:
check_value("rfe_features", rfe_features, "5e2ad2d9", "cele trei nume, sortate alfabetic \u0219i unite cu \",\"")
check_value("bacc_rfe_inside", bacc_rfe_inside, "119e73ca", "RFE \u00een interiorul pipeline-ului (reantrenat \u00een fiecare fold de antrenare)")
check_value("bacc_rfe_outside", bacc_rfe_outside, "b8cbc389", "acelea\u0219i 3 tr\u0103s\u0103turi alese O SINGUR\u0102 DAT\u0102 pe toate zilele, apoi evaluate")
check("selecția în afara CV arată mai bine decât cea cinstită", bacc_rfe_outside > bacc_rfe_inside, "ziua de test a influențat alegerea")

**Interpretarea voastră:** _scrieți aici_

## 3.9 · Lasso față de RFE  ★★☆

*(din laboratorul inițial, Exercițiul 3 din secțiunea 2.5)* Antrenați `Lasso(alpha=0.1)` pe tabelul real **standardizat** (scădeți media și împărțiți la abaterea standard a fiecărei coloane) cu **numărul de persoane** `cnt` ca țintă numerică:

* `n_lasso` – numărul de coeficienți nenuli; `lasso_features` – numele lor, sortate și unite cu `","`;
* `n_common` – câte dintre ele se află și printre cele trei trăsături alese de RFE în 3.8 (`sel`).

**Scrieți:** de ce două metode aleg trăsături diferite? În care metodă ați avea mai multă încredere și ce verificare suplimentară (3.5, 3.6) ați face înainte de a interpreta oricare dintre liste?

In [ ]:
Xo, cnt, dayo = load_occupancy(); yo = (cnt > 0.5).astype(int)
lodo = LeaveOneGroupOut(); LR = lambda: LogisticRegression(max_iter=3000)
f = lambda m, cols: bacc(yo, cross_val_predict(m, Xo[cols], yo, cv=lodo, groups=dayo))

Z = StandardScaler().fit_transform(Xo)
sel = list(Xo.columns[RFE(LR(), n_features_to_select=3).fit(Z, yo).support_])           # din 3.8
from sklearn.linear_model import Lasso
n_lasso = ...
lasso_features = ...
n_common = ...

**Autoverificare**

In [ ]:
check_value("n_lasso", n_lasso, "9ae1524b", "coeficien\u021bi nenuli, Lasso(alpha=0,1)")
check_value("lasso_features", lasso_features, "980117c6", "numele selectate, sortate \u0219i unite cu \",\"")
check_value("n_common", n_common, "9f770654", "tr\u0103s\u0103turi selectate \u0219i de Lasso, \u0219i de RFE din 3.8")
check("cele două metode sunt de acord asupra cel puțin unei trăsături", n_common >= 1, "")

**Interpretarea voastră:** _scrieți aici_

# Partea 4 — Primul vostru model

Se folosesc două tabele: tabelul **biroului** (ferestre de 15 minute) și tabelul **ocupării reale** `Xo, cnt, dayo = load_occupancy()` (ferestre de 5 minute). Pentru datele reale excludem senzorul de mișcare în majoritatea sarcinilor: `no_pir = [c for c in Xo.columns if not c.startswith("PIR")]`.

## 4.1 · Linii de bază pe datele reale  ★☆☆

Cu toate cele 24 de trăsături ale tabelului real (lasă-o-zi-afară):

* `acc_major`, `bacc_major` – acuratețea și acuratețea echilibrată ale lui `DummyClassifier(strategy="most_frequent")`;
* `best_single` – **numele** trăsăturii al cărei prag pe o singură trăsătură (`DecisionTreeClassifier(max_depth=1, random_state=0)`) are cea mai mare acuratețe echilibrată; `bacc_single` – acel scor.

**Gândiți-vă:** ce este suspect la cea mai bună trăsătură? Ce întrebare ați pune celor care au instalat senzorii?

In [ ]:
Xo, cnt, dayo = load_occupancy(); yo = (cnt > 0.5).astype(int)
no_pir = [c for c in Xo.columns if not c.startswith("PIR")]
lodo = LeaveOneGroupOut()

def held(model, Xm, y=yo): return cross_val_predict(model, Xm, y, cv=lodo, groups=dayo)
pred = held(DummyClassifier(strategy="most_frequent"), Xo)
acc_major = ...
bacc_major = ...
best_single = ...
bacc_single = ...

**Autoverificare**

In [ ]:
check_value("acc_major", acc_major, "b8991633", "acurate\u021bea r\u0103spunsului constant cu clasa majoritar\u0103 (las\u0103-o-zi-afar\u0103)")
check_value("bacc_major", bacc_major, "f88afaea", "acurate\u021bea echilibrat\u0103 a aceluia\u0219i r\u0103spuns")
check_value("best_single", best_single, "d6807b60", "numele tr\u0103s\u0103turii cu cel mai bun prag pe o singur\u0103 tr\u0103s\u0103tur\u0103 (arbore de ad\u00e2ncime 1, toate cele 24 de tr\u0103s\u0103turi)")
check_value("bacc_single", bacc_single, "b45898ec", "acurate\u021bea ei echilibrat\u0103")

**Interpretarea voastră:** _scrieți aici_

## 4.2 · Antrenați, evaluați, diagnosticați  ★★☆

Pe `Xo[no_pir]` și eticheta `yo`, antrenați **StandardScaler + LogisticRegression(max_iter=3000)**:

* `held_lr` – acuratețea echilibrată pe date nevăzute cu lasă-o-zi-afară;
* `train_lr` – acuratețea echilibrată medie de **antrenare** pe cele 7 fold-uri (antrenat pe 6 zile, evaluat pe aceleași 6 zile);
* `diagnosis` – șirul `"overfitting"` dacă `train_lr - held_lr > 0.02`, altfel `"ok"`.

In [ ]:
Xo, cnt, dayo = load_occupancy(); yo = (cnt > 0.5).astype(int)
no_pir = [c for c in Xo.columns if not c.startswith("PIR")]
lodo = LeaveOneGroupOut()

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
held_lr = ...
train_lr = ...
diagnosis = ...

**Autoverificare**

In [ ]:
check_value("held_lr", held_lr, "119e73ca", "acurate\u021bea echilibrat\u0103 pe date nev\u0103zute (las\u0103-o-zi-afar\u0103)")
check_value("train_lr", train_lr, "b45898ec", "acurate\u021bea echilibrat\u0103 medie pe antrenare, pe cele 7 fold-uri")
check_value("diagnosis", diagnosis, "a1f32f57", "\"overfitting\" dac\u0103 train - held > 0,02, altfel \"ok\"")

**Interpretarea voastră:** _scrieți aici_

## 4.3 · Adâncimea arborelui  ★★☆

Pe `Xo[no_pir]`, evaluați `DecisionTreeClassifier(max_depth=k, random_state=0)` pentru `k = 1 … 8` cu lasă-o-zi-afară (acuratețe echilibrată).

* `best_depth` – adâncimea cu cel mai mare scor pe date nevăzute (cea **mai mică** astfel de adâncime, în caz de egalitate);
* `chosen_depth` – cea **mai mică** adâncime al cărei scor este la cel mult 0,005 de cel mai bun.

**Scrieți:** de ce cel mai simplu model de pe platou este o alegere mai bună decât maximul absolut?

In [ ]:
Xo, cnt, dayo = load_occupancy(); yo = (cnt > 0.5).astype(int)
no_pir = [c for c in Xo.columns if not c.startswith("PIR")]
lodo = LeaveOneGroupOut()

scores = {}
for k in range(1, 9):
    ...
best_depth = ...
chosen_depth = ...

**Autoverificare**

In [ ]:
check_value("best_depth", best_depth, "9f770654", "ad\u00e2ncimea cu cel mai mare scor pe date nev\u0103zute")
check_value("chosen_depth", chosen_depth, "9f770654", "cea mai MIC\u0102 ad\u00e2ncime al c\u0103rei scor este la cel mult 0,005 de cel mai bun")
check("ați ales un arbore simplu", chosen_depth <= best_depth, "")

**Interpretarea voastră:** _scrieți aici_

## 4.4 · Intensitatea regularizării  ★★☆

Pe `Xo[no_pir]` cu **StandardScaler + LogisticRegression(C=C, max_iter=3000)** și lasă-o-zi-afară, raportați acuratețea echilibrată pe date nevăzute pentru `C = 0.0001` (`C_small`) și `C = 1` (`C_one`).

**Gândiți-vă:** ce prezice modelul pentru *fiecare* fereastră când `C` este atât de mic? De ce acuratețea echilibrată este exact 0,5?

In [ ]:
Xo, cnt, dayo = load_occupancy(); yo = (cnt > 0.5).astype(int)
no_pir = [c for c in Xo.columns if not c.startswith("PIR")]
lodo = LeaveOneGroupOut()

def at(C):
    ...
C_small, C_one = at(0.0001), at(1)

**Autoverificare**

In [ ]:
check_value("C_small", C_small, "f88afaea", "scorul pe date nev\u0103zute la C = 0,0001")
check_value("C_one", C_one, "119e73ca", "scorul pe date nev\u0103zute la C = 1")
check("o penalizare prea puternică subantrenează", C_small < C_one - 0.05, "")

**Interpretarea voastră:** _scrieți aici_

## 4.5 · Citiți matricea de confuzie  ★☆☆

Pe tabelul **biroului**, evaluați **StandardScaler + LogisticRegression(max_iter=3000)** cu lasă-o-zi-afară și extrageți `tn, fp, fn, tp` din `confusion_matrix(y, pred).ravel()`; calculați `recall_occ`.

**Gândiți-vă:** pentru un controler de ventilație, care eroare este mai gravă — o alarmă falsă sau o ratare?

In [ ]:
d = load_office(); cols = ["temp_c", "humidity_pct", "co2_ppm"]
X, yp, day = window_table(d, cols, 15, 15); y = (yp > 0.5).astype(int)
lodo = LeaveOneGroupOut()

pred = ...
tn, fp, fn, tp = ...
recall_occ = ...

**Autoverificare**

In [ ]:
check_value("tn", tn, "174d5002", "gol numit gol")
check_value("fp", fp, "b2337ecb", "gol numit ocupat")
check_value("fn", fn, "a97491cd", "ocupat numit gol")
check_value("tp", tp, "9ea7379a", "ocupat numit ocupat")
check_value("recall_occ", recall_occ, "b45898ec", "tp / (tp + fn)")

**Interpretarea voastră:** _scrieți aici_

## 4.6 · Regresie: câte persoane sunt?  ★★☆

Tabelul biroului, ținta `yp` (numărul mediu de persoane), lasă-o-zi-afară, MAE în persoane:

* `mae_mean` (mereu media), `mae_ridge` (**StandardScaler + Ridge(alpha=1.0)**), `mae_tree` (`DecisionTreeRegressor(max_depth=3, random_state=0)`);
* `ratio_best` – cel mai mic MAE împărțit la `mae_mean` (sub 1 = mai bun decât linia de bază).

In [ ]:
d = load_office(); cols = ["temp_c", "humidity_pct", "co2_ppm"]
X, yp, day = window_table(d, cols, 15, 15); y = (yp > 0.5).astype(int)
lodo = LeaveOneGroupOut()

mae_mean = ...
mae_ridge = ...
mae_tree = ...
ratio_best = ...

**Autoverificare**

In [ ]:
check_value("mae_mean", mae_mean, "41462174", "mereu media (persoane)")
check_value("mae_ridge", mae_ridge, "b440d5b1", "Ridge(alpha=1) pe tr\u0103s\u0103turi standardizate")
check_value("mae_tree", mae_tree, "6d49251b", "DecisionTreeRegressor(max_depth=3, random_state=0)")
check_value("ratio_best", ratio_best, "37587849", "cel mai bun MAE / MAE-ul liniei de baz\u0103")

**Interpretarea voastră:** _scrieți aici_

## 4.7 · O linie de bază pentru prognoză  ★★★

Ținta: **media CO₂ a ferestrei următoare**. Construiți-o din tabelul biroului: `target = X["co2_mean"].shift(-1)`; eliminați ultimul rând (nu există fereastră următoare). Păstrați grupul `day` aliniat. Cu lasă-o-zi-afară:

* `mae_mean_f` – linia de bază „mereu media” (`DummyRegressor`);
* `mae_persist` – **persistența**: prezice valoarea următoare ca `co2_mean` curent (fără model);
* `mae_ridge_f` – **StandardScaler + Ridge(alpha=1.0)** pe toate cele 36 de trăsături curente.

**Gândiți-vă:** de ce persistența este atât de greu de bătut pentru un semnal care variază lent și de ce „mereu media” este o linie de bază proastă pentru o prognoză? (Cursul 8 dezvoltă această idee.)

In [ ]:
d = load_office(); cols = ["temp_c", "humidity_pct", "co2_ppm"]
X, yp, day = window_table(d, cols, 15, 15); y = (yp > 0.5).astype(int)
lodo = LeaveOneGroupOut()

target = X["co2_mean"].shift(-1).iloc[:-1].values
Xf, gf = X.iloc[:-1], day[:-1]
mae_mean_f = ...
mae_persist = ...
mae_ridge_f = ...

**Autoverificare**

In [ ]:
check_value("mae_mean_f", mae_mean_f, "11b39e7c", "mereu media zilelor de antrenare")
check_value("mae_persist", mae_persist, "344eaa7b", "prezice fereastra urm\u0103toare cu media ferestrei CURENTE")
check_value("mae_ridge_f", mae_ridge_f, "fd68be7a", "ridge pe tr\u0103s\u0103turile curente")
check("persistența bate media la mare distanță", mae_persist < mae_mean_f / 5, "")

**Interpretarea voastră:** _scrieți aici_

## 4.8 · Cât de mult flatează o împărțire aleatorie?  ★★☆

Pe `Xo[no_pir]` cu **StandardScaler + LogisticRegression(max_iter=3000)**: `acc_rand` – acuratețea echilibrată cu `KFold(5, shuffle=True, random_state=0)`; `acc_group` – cu lasă-o-zi-afară.

**Scrieți:** pe care dintre cele două numere l-ați promite unui client care va instala sistemul într-o cameră nouă? De ce?

In [ ]:
Xo, cnt, dayo = load_occupancy(); yo = (cnt > 0.5).astype(int)
no_pir = [c for c in Xo.columns if not c.startswith("PIR")]
lodo = LeaveOneGroupOut()

m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
acc_rand = ...
acc_group = ...

**Autoverificare**

In [ ]:
check_value("acc_rand", acc_rand, "b8cbc389", "5 fold-uri aleatorii (shuffle, random_state=0)")
check_value("acc_group", acc_group, "119e73ca", "las\u0103-o-zi-afar\u0103")
check("împărțirea aleatorie flatează modelul (> 0,02)", acc_rand - acc_group > 0.02, "")

**Interpretarea voastră:** _scrieți aici_

# Partea 5 — Mini-proiect: un tabel de rezultate pentru datele reale de ocupare

Aceasta este **tema**: puneți totul cap la cap în câteva celule și scrieți concluziile. Verificările automate controlează doar că tabelul există și are structura corectă — *calitatea* lucrării este în textul vostru.

## 5.1 · Tabel de rezultate cu linii de bază, modele și variație  ★★★

Folosind datele reale de ocupare **fără senzorul PIR** (`Xo[no_pir]`, eticheta `yo`, grupurile `dayo`):

1. Construiți un `DataFrame results` cu **un rând pentru fiecare metodă** și coloanele `model`, `features` (câte trăsături), `bacc_mean`, `bacc_std`. Media și abaterea standard sunt calculate pe acuratețea echilibrată **pe fiecare zi** din lasă-o-zi-afară, **doar pentru zilele care conțin ambele clase** (zilele cu o singură clasă nu au acuratețe echilibrată).
2. Rânduri obligatorii: **linia de bază majoritară**, **cel mai bun prag pe o singură trăsătură**, o **regresie logistică** (cu un `C` ales pe baza dovezilor), un **arbore de decizie** (cu o adâncime aleasă pe baza dovezilor) și **încă un model sau subset de trăsături** la alegere (de exemplu RFE, L1, pădure aleatorie). Toată selecția și scalarea **în interiorul pipeline-urilor**.
3. Sub tabel, într-o celulă markdown, scrieți **cinci propoziții**: ce metodă ați pune în funcțiune și de ce; efectul eliminării PIR; numărul de zile; un risc al evaluării; ce ați face cu încă 30 de zile.

**Predare:** *Kernel → Restart & Run All*, toate ✅, plus textul vostru. Numiți fișierul `Lab5_Prenume_Nume.ipynb`.

In [ ]:
Xo, cnt, dayo = load_occupancy(); yo = (cnt > 0.5).astype(int)
no_pir = [c for c in Xo.columns if not c.startswith("PIR")]
lodo = LeaveOneGroupOut()

results = pd.DataFrame(columns=["model", "features", "bacc_mean", "bacc_std"])
# ... construiți aici câte un rând pentru fiecare metodă, de exemplu cu o funcție care returnează acuratețea echilibrată pe fiecare zi ...

**Autoverificare**

In [ ]:
check("`results` este un DataFrame", isinstance(results, pd.DataFrame), "construiți un DataFrame pandas")
check("cel puțin 5 rânduri (linii de bază + modele)", len(results) >= 5, "majoritar, cea mai bună trăsătură singură, + cel puțin 3 modele")
check("coloane: model, features, bacc_mean, bacc_std", {"model", "features", "bacc_mean", "bacc_std"} <= set(results.columns), "")
check("există o linie de bază majoritară (scor 0,5)", (results.bacc_mean.round(2) == 0.5).any(), "")
check("scorurile sunt valide", results.bacc_mean.between(0, 1).all(), "")

**Interpretarea voastră:** _scrieți aici_

# Partea 6 — Un al doilea set de date: Smart Home Energy Usage

*(din laboratorul inițial, Exercițiul 4 din secțiunile 1.5 și 2.5)* `smart_home_energy_usage_dataset.csv` are un rând pe oră cu `energy_consumption_kWh`, setarea de temperatură, starea de ocupare, aparatul, anotimpul, ziua săptămânii și un indicator de sărbătoare. **Citiți doar primele 50 000 de rânduri** (`nrows=50000`: fișierul are un milion de rânduri, adică aproximativ 114 „ani”).

Setul de date este **sintetic**. Înainte de a construi orice, se aplică întrebarea întregului laborator: *există vreo structură de găsit?* Cele două exerciții de mai jos sunt un test cinstit; concluzia corectă poate fi una **negativă**, iar un rezultat negativ, susținut de o linie de bază, este un rezultat valid.

## 6.1 · Extragerea trăsăturilor din consumul de energie  ★★☆

*(Exercițiul 4 inițial, partea 1: trăsături statistice, Fourier și wavelet ale consumului de energie)* Luați **primele 8760 de valori** din `energy_consumption_kWh` (un an, o valoare pe oră).

* **Statistici:** `mean_kwh`, `std_kwh` (`ddof=1`).
* **Fourier:** scădeți media, calculați spectrul de putere `|rfft|²` și frecvențele în cicluri pe oră (`np.fft.rfftfreq(8760, d=1)`). `period_dom_h` = perioada (în ore) a celui mai mare vârf (f > 0); `peak_ratio` = cea mai mare putere / puterea **mediană**; `p24_ratio` = puterea la canalul perioadei de 24 de ore (`np.argmin(abs(freq - 1/24))`) împărțită la puterea mediană; `has_daily_cycle` = `True` dacă `p24_ratio > 10`.
* **Wavelet:** `pywt.wavedec(x, "db4", level=4)`; `frac_cD1` = energia ultimului vector (cel mai fin detaliu, banda 0,5–1) împărțită la energia totală.

**Scrieți:** găsiți cicluri zilnice sau săptămânale, vârfuri de dimineață sau de seară? Pentru un semnal de zgomot alb, energia celui mai fin nivel de detaliu este aproximativ **jumătate** din total, următorul un sfert și așa mai departe — arată rezultatul vostru wavelet așa? Cum arată raportul `peak_ratio` al unui semnal *aleatoriu* și de ce un cel mai mare vârf de 12 ori mediana nu este dovada unui ciclu?

In [ ]:
h = pd.read_csv("smart_home_energy_usage_dataset.csv", nrows=50000, parse_dates=["timestamp"])

y = h["energy_consumption_kWh"].values[:8760]
mean_kwh = ...
std_kwh = ...
x = y - y.mean(); spec = np.abs(np.fft.rfft(x)) ** 2; freq = np.fft.rfftfreq(len(x), d=1)
period_dom_h = ...
peak_ratio = ...
p24_ratio = ...
has_daily_cycle = ...
import pywt
frac_cD1 = ...

**Autoverificare**

In [ ]:
check_value("mean_kwh", mean_kwh, "30099f80", "media primelor 8760 de valori orare (un an)")
check_value("std_kwh", std_kwh, "9909ad1a", "abaterea standard cu ddof=1")
check_value("period_dom_h", period_dom_h, "2687b468", "1 / frecven\u021ba celui mai mare v\u00e2rf spectral, \u00een ore")
check_value("peak_ratio", peak_ratio, "87c13e35", "putere maxim\u0103 / putere median\u0103 (f > 0)")
check_value("p24_ratio", p24_ratio, "004a9e08", "puterea la canalul perioadei de 24 de ore / puterea median\u0103")
check_value("frac_cD1", frac_cD1, "db964088", "energia celui mai fin nivel de detaliu / energia total\u0103 (db4, 4 niveluri)")
check_value("has_daily_cycle", has_daily_cycle, "60a33e6c", "True dac\u0103 p24_ratio > 10, altfel False")
check("spectrul arată ca zgomot (peak_ratio < 20)", peak_ratio < 20, "")

**Interpretarea voastră:** _scrieți aici_

## 6.2 · Selecția trăsăturilor pentru prezicerea energiei  ★★★

*(Exercițiul 4 inițial, partea de selecție: corelație, RFE și Lasso pentru prezicerea consumului)* Construiți tabelul de trăsături `X` din primele 50 000 de rânduri: `temperature_setting_C`, `usage_duration_minutes`, `holiday`, `home_id`, **ora din zi** și coloanele one-hot ale `occupancy_status`, `appliance`, `season`, `day_of_week` (`pd.get_dummies(..., dtype=float)`); ținta este `energy_consumption_kWh`. Împărțiți **în timp**: primele 70 % din rânduri pentru antrenare, ultimele 30 % pentru test; standardizați cu statisticile de antrenare.

* `n_features` – numărul de coloane ale lui `X`; `max_abs_corr` – cea mai mare corelație Pearson absolută cu ținta;
* `n_lasso` – coeficienții nenuli ai `Lasso(alpha=0.1)` antrenat pe partea de antrenare;
* `mae_base` – MAE pe test al lui „mereu media de antrenare”; `mae_model` – MAE pe test al unei `LinearRegression` care folosește cele **5 trăsături** alese de `RFE(LinearRegression(), n_features_to_select=5)` pe partea de antrenare; `ratio = mae_model / mae_base`; `predictable` = `True` dacă `ratio < 0.95`.

**Scrieți:** scrieți concluzia pe care ați da-o unui client care a întrebat „ce factori îmi determină consumul?”. Ce înseamnă corelația de 0,009 și ce ați verifica înainte de a colecta mai multe date?

In [ ]:
h = pd.read_csv("smart_home_energy_usage_dataset.csv", nrows=50000, parse_dates=["timestamp"])

from sklearn.linear_model import Lasso, LinearRegression
X = pd.get_dummies(h.drop(columns=["timestamp", "energy_consumption_kWh"]).assign(hour=h.timestamp.dt.hour),
                   columns=["occupancy_status", "appliance", "season", "day_of_week"], dtype=float)
y = h["energy_consumption_kWh"].values; n = int(0.7 * len(X))
mu, sd = X.iloc[:n].mean(), X.iloc[:n].std(); Xs = (X - mu) / sd
n_features = ...
max_abs_corr = ...
n_lasso = ...
mae_base = ...
mae_model = ...
ratio = ...
predictable = ...

**Autoverificare**

In [ ]:
check_value("n_features", n_features, "85d40e1b", "coloane dup\u0103 codificarea one-hot + ora")
check_value("max_abs_corr", max_abs_corr, "312b95ee", "cea mai mare |corela\u021bie| cu \u021binta printre cele 24 de coloane")
check_value("n_lasso", n_lasso, "561b2814", "coeficien\u021bi Lasso(alpha=0,1) nenuli (antrenat pe primele 70 %)")
check_value("mae_base", mae_base, "5dcdf9da", "MAE pe ultimele 30 % pentru \u201emereu media de antrenare\u201d")
check_value("mae_model", mae_model, "5dcdf9da", "MAE al regresiei liniare pe cele 5 tr\u0103s\u0103turi RFE")
check_value("ratio", ratio, "cf9dcf6d", "mae_model / mae_base")
check_value("predictable", predictable, "60a33e6c", "True dac\u0103 ratio < 0,95, altfel False")
check("niciun câștig față de linia de bază (ratio > 0,98)", ratio > 0.98, "")

**Interpretarea voastră:** _scrieți aici_